In [14]:

# Import the libraries
import pandas as pd
from pathlib import Path

In [15]:
# Define the absolute path to the completed audit CSV
AUDIT_PATH = Path(
    "/Users/laasyavavilapalli/Downloads/research_agentic-table-qa/"
    "anacodic-agentic-cookbook/02-experiments/"
    "laasya-table-eval/results/audit_sheet - audit_sheet.csv.csv"
)

# Verify that the audit file exists before attempting to load it
assert AUDIT_PATH.exists(), f"Audit file not found: {AUDIT_PATH}"

# Load the completed manual audit into a DataFrame
df = pd.read_csv(AUDIT_PATH)

# Display the first five rows to verify successful loading
df.head()

,item_id,question_short,gold,gold_in_table__yes_no_partial,gold_looks_correct__yes_no_unsure,why_model_wrong__category,notes_optional
0,item_001,What percentage of oocytes tested in the categ...,6.66%,yes,yes,NaN,NaN
1,item_002,How many oocytes with Simple Errors were contr...,1,yes,yes,NaN,NaN
2,item_003,What percentage of oocytes tested had Simple E...,0%,yes,yes,NaN,NaN
3,item_004,How many oocytes tested had premeiotic aneuplo...,25,yes,yes,NaN,NaN
4,item_005,What is the average Effect value for all entri...,-0.06,no,no,label_wrong,"gold value is -0.06, table value is -0.045, mo..."


In [16]:
# Display the total number of audited questions
print("Number of audit questions:", len(df))

# Display the total number of columns
print("Number of columns:", len(df.columns))

# Inspect the available audit fields
print("\nColumn names:")
print(df.columns.tolist())

Number of audit questions: 60
Number of columns: 7

Column names:
['item_id', 'question_short', 'gold', 'gold_in_table__yes_no_partial', 'gold_looks_correct__yes_no_unsure', 'why_model_wrong__category', 'notes_optional']


In [17]:
# Count the number of questions in each answerability category
answerability_counts = df[
    "gold_in_table__yes_no_partial"
].value_counts(dropna=False)

# Display the resulting distribution
print("Answerability Distribution:")
print(answerability_counts)

Answerability Distribution:
gold_in_table__yes_no_partial
yes        46
no         10
partial     3
NaN         1
Name: count, dtype: int64


In [18]:
# Calculate the percentage represented by each answerability category
answerability_percentages = (
    answerability_counts / len(df) * 100
)

# Round percentages to two decimal places for reporting
answerability_percentages = answerability_percentages.round(2)

# Display the percentage distribution
print("Answerability Percentages:")
print(answerability_percentages)

Answerability Percentages:
gold_in_table__yes_no_partial
yes        76.67
no         16.67
partial     5.00
NaN         1.67
Name: count, dtype: float64


In [19]:
# Count the gold-answer correctness classifications
gold_correctness_counts = df[
    "gold_looks_correct__yes_no_unsure"
].value_counts(dropna=False)

# Calculate the percentage of all 60 audit items in each category
gold_correctness_percentages = (
    gold_correctness_counts / len(df) * 100
).round(2)

# Combine counts and percentages into one summary DataFrame
gold_correctness_summary = pd.DataFrame({
    "Count": gold_correctness_counts,
    "Percentage": gold_correctness_percentages
})

# Display the gold-answer correctness distribution
print("Gold Answer Correctness:")
display(gold_correctness_summary)

Gold Answer Correctness:


,Count,Percentage
gold_looks_correct__yes_no_unsure,,
yes,46,76.67
no,10,16.67
unsure,3,5.00
NaN,1,1.67


In [20]:
# Count the recorded model error categories, including missing values
error_counts = df[
    "why_model_wrong__category"
].value_counts(dropna=False)

# Calculate percentages using all 60 audited questions
error_percentages = (
    error_counts / len(df) * 100
).round(2)

# Combine the counts and percentages into a summary table
error_summary = pd.DataFrame({
    "Count": error_counts,
    "Percentage": error_percentages
})

# Display the error-category distribution
print("Model Error Category Distribution:")
display(error_summary)

Model Error Category Distribution:


,Count,Percentage
why_model_wrong__category,,
NaN,44,73.33
label_wrong,12,20.00
wrong_cell,3,5.00
other,1,1.67


In [21]:
# Create a list of summary statistics for each audit measure
summary_data = [
    {
        "Measure": "Table answerability",
        "Classification": category,
        "Count": count,
        "Percentage": round(count / len(df) * 100, 2)
    }
    for category, count in answerability_counts.items()
]

# Add gold-answer correctness results to the summary
summary_data.extend([
    {
        "Measure": "Gold answer correctness",
        "Classification": category,
        "Count": count,
        "Percentage": round(count / len(df) * 100, 2)
    }
    for category, count in gold_correctness_counts.items()
])

# Add model error-category results to the summary
summary_data.extend([
    {
        "Measure": "Model error category",
        "Classification": category,
        "Count": count,
        "Percentage": round(count / len(df) * 100, 2)
    }
    for category, count in error_counts.items()
])

# Convert the collected statistics into a DataFrame
audit_summary = pd.DataFrame(summary_data)

# Display the combined audit summary
display(audit_summary)

,Measure,Classification,Count,Percentage
0,Table answerability,yes,46,76.67
1,Table answerability,no,10,16.67
2,Table answerability,partial,3,5.00
3,Table answerability,NaN,1,1.67
4,Gold answer correctness,yes,46,76.67
5,Gold answer correctness,no,10,16.67
6,Gold answer correctness,unsure,3,5.00
7,Gold answer correctness,NaN,1,1.67
8,Model error category,NaN,44,73.33
9,Model error category,label_wrong,12,20.00


In [22]:
# Define the intended results directory in the Downloads project
RESULTS_DIR = Path(
    "/Users/laasyavavilapalli/Downloads/research_agentic-table-qa/"
    "anacodic-agentic-cookbook/02-experiments/"
    "laasya-table-eval/results"
)

# Verify that the target directory exists
assert RESULTS_DIR.is_dir(), f"Directory not found: {RESULTS_DIR}"

# Construct the output file path
SUMMARY_PATH = RESULTS_DIR / "audit_summary.csv"

# Export the audit summary to the intended project
audit_summary.to_csv(SUMMARY_PATH, index=False)

# Verify the exact output location
print("Saved to:", SUMMARY_PATH.resolve())
print("File exists:", SUMMARY_PATH.exists())

Saved to: /Users/laasyavavilapalli/Downloads/research_agentic-table-qa/anacodic-agentic-cookbook/02-experiments/laasya-table-eval/results/audit_summary.csv
File exists: True


In [23]:
# Compare answerability classifications with recorded model error categories
error_by_answerability = pd.crosstab(
    df["gold_in_table__yes_no_partial"].fillna("missing"),
    df["why_model_wrong__category"].fillna("missing")
)

# Display the cross-tabulation
display(error_by_answerability)

why_model_wrong__category,label_wrong,missing,other,wrong_cell
gold_in_table__yes_no_partial,,,,
missing,0,1,0,0
no,10,0,0,0
partial,2,1,0,0
yes,0,42,1,3


In [24]:
# Select answerable questions without a recorded model error category
missing_error_examples = df[
    (df["gold_in_table__yes_no_partial"] == "yes") &
    (df["why_model_wrong__category"].isna())
]

# Display the question, gold answer, and auditor notes
display(
    missing_error_examples[
        ["item_id", "question_short", "gold", "notes_optional"]
    ]
)

,item_id,question_short,gold,notes_optional
0,item_001,What percentage of oocytes tested in the categ...,6.66%,NaN
1,item_002,How many oocytes with Simple Errors were contr...,1,NaN
2,item_003,What percentage of oocytes tested had Simple E...,0%,NaN
3,item_004,How many oocytes tested had premeiotic aneuplo...,25,NaN
5,item_006,What is the percentage of oocytes with premeio...,14.50%,NaN
6,item_007,How many oocytes with Simple Errors were contr...,3,NaN
8,item_009,What is the forward primer sequence for Exon 3...,ACGTAGTGCATACACCCTTG,NaN
9,item_010,What is the reverse primer sequence for Exon 2?,GGAAAAGCCAAATCTAAGGTTCC,NaN
10,item_011,Which region has the forward primer sequence '...,Promoter & Exon 1,NaN
11,item_012,What is the forward primer sequence for Exon 5?,GAAACAGGTTTCCTTTCCCAAG,NaN


44/60 (73.33%)
All models correct

16/60 (26.67%)
At least one model marked wrong

All 16 recorded error cases were classified as label_wrong.

In [25]:
# Compare whether the gold answer is supported by the table
# against the auditor's judgment of whether the gold is correct
gold_validity_crosstab = pd.crosstab(
    df["gold_in_table__yes_no_partial"].fillna("missing"),
    df["gold_looks_correct__yes_no_unsure"].fillna("missing")
)

# Display the relationship between the two audit judgments
display(gold_validity_crosstab)

gold_looks_correct__yes_no_unsure,missing,no,unsure,yes
gold_in_table__yes_no_partial,,,,
missing,1,0,0,0
no,0,10,0,0
partial,0,0,3,0
yes,0,0,0,46


In [27]:
# Summarize the three human-audit dimensions independently
audit_measures = {
    "Table answerability": "gold_in_table__yes_no_partial",
    "Gold answer correctness": "gold_looks_correct__yes_no_unsure",
    "Model error category": "why_model_wrong__category"
}

# Collect counts and percentages for each audit dimension
summary_data = []

for measure, column in audit_measures.items():
    counts = df[column].value_counts(dropna=False)

    for classification, count in counts.items():
        # Label blank classifications explicitly for readability
        if pd.isna(classification):
            classification = "blank"

        summary_data.append({
            "Measure": measure,
            "Classification": classification,
            "Count": count,
            "Percentage": round(count / len(df) * 100, 2)
        })

# Create the final summary DataFrame
audit_summary = pd.DataFrame(summary_data)

# Display the final audit summary
display(audit_summary)

,Measure,Classification,Count,Percentage
0,Table answerability,yes,46,76.67
1,Table answerability,no,10,16.67
2,Table answerability,partial,3,5.00
3,Table answerability,blank,1,1.67
4,Gold answer correctness,yes,46,76.67
5,Gold answer correctness,no,10,16.67
6,Gold answer correctness,unsure,3,5.00
7,Gold answer correctness,blank,1,1.67
8,Model error category,blank,44,73.33
9,Model error category,label_wrong,12,20.00


In [28]:
# Define the results directory in the research project
RESULTS_DIR = Path(
    "/Users/laasyavavilapalli/Downloads/research_agentic-table-qa/"
    "anacodic-agentic-cookbook/02-experiments/"
    "laasya-table-eval/results"
)

# Save the final summary
SUMMARY_PATH = RESULTS_DIR / "audit_summary.csv"
audit_summary.to_csv(SUMMARY_PATH, index=False)

# Verify the saved file
print("Saved to:", SUMMARY_PATH.resolve())
print("File exists:", SUMMARY_PATH.exists())

Saved to: /Users/laasyavavilapalli/Downloads/research_agentic-table-qa/anacodic-agentic-cookbook/02-experiments/laasya-table-eval/results/audit_summary.csv
File exists: True


In [29]:
# Define the results directory in the research project
RESULTS_DIR = Path(
    "/Users/laasyavavilapalli/Downloads/research_agentic-table-qa/"
    "anacodic-agentic-cookbook/02-experiments/"
    "laasya-table-eval/results"
)

# Define the final summary output path
SUMMARY_PATH = RESULTS_DIR / "audit_summary.csv"

# Export the final audit summary
audit_summary.to_csv(SUMMARY_PATH, index=False)

# Verify the output location and file creation
print("Saved to:", SUMMARY_PATH.resolve())
print("File exists:", SUMMARY_PATH.exists())

Saved to: /Users/laasyavavilapalli/Downloads/research_agentic-table-qa/anacodic-agentic-cookbook/02-experiments/laasya-table-eval/results/audit_summary.csv
File exists: True


### Manual Audit Results

A manual audit was conducted on 60 questions to evaluate table answerability, gold-answer correctness, and model-error categories.

**Table answerability:** Of the 60 questions, 46 (76.67%) were classified as answerable from the provided tables, 10 (16.67%) as unanswerable, and 3 (5.00%) as partially answerable. One question (1.67%) had no recorded classification.

**Gold-answer correctness:** The gold answers for 46 questions (76.67%) were judged correct, while 10 (16.67%) were judged incorrect and 3 (5.00%) were marked unsure. One question (1.67%) had no recorded judgment.

**Model-error categories:** In 44 questions (73.33%), all models were judged correct, so no error category was recorded. Among the remaining 16 questions, 12 (20.00% of all audited questions) were classified as `label_wrong`, 3 (5.00%) as `wrong_cell`, and 1 (1.67%) as `other`.

These results summarize the recorded human-audit judgments and should not be interpreted as standalone model accuracy measurements.